\setcounter{secnumdepth}{-1}

# Parameter-Efficient Fine-Tuning Study
## LoRA Rank Ablation on GPT-2 for Dialogue Summarization

**Author:** Chris Schmidt, AI Engineering MSE Candidate (Johns Hopkins University)  
**Date:** April 2026  
**Repository:** [github.com/PCSchmidt/finetuning-study](https://github.com/PCSchmidt/finetuning-study)

---

## Abstract

Large language models are expensive to fine-tune. GPT-2, one of the smaller publicly available models, still has 124 million parameters, and modern models have tens or hundreds of billions. Updating all of those weights for a specific task requires significant compute, memory, and time. Low-Rank Adaptation, or LoRA, offers a practical alternative: instead of updating every weight in the model, LoRA freezes the original pretrained weights entirely and injects small trainable matrices into specific layers. These adapter matrices are a fraction of the size of the full model, which means training is faster, uses less memory, and can even run on a CPU.

This study applies LoRA to GPT-2 (124M parameters) for dialogue summarization using the SAMSum dataset, a collection of messenger-style conversations paired with human-written summaries. The central question is: **how does the rank of the LoRA adapter, which controls how many independent dimensions the adapter can learn in, affect the quality of the fine-tuned model?**

To answer that, we run a systematic ablation over five LoRA configurations, varying rank across [2, 4, 8, 16] and the scaling parameter alpha across [8, 16, 32]. Each configuration is trained for three epochs on a 500-example subset of SAMSum, with 100 validation and 100 test examples. We evaluate using ROUGE (which measures word-level overlap between generated and reference summaries) and BERTScore (which measures semantic similarity using contextual embeddings). All training runs on CPU only, which is a deliberate choice: the goal is to show that meaningful fine-tuning research is feasible on commodity hardware without GPU access.

This notebook is a personal learning project completed as part of coursework in the Johns Hopkins AI Engineering MSE program. It is not production work, but it is designed with the same rigor you would bring to a real experiment: reproducible results, version-controlled code, and honest reporting of what worked and what did not.

---

## Contents

1. [Motivation](#1-motivation)
2. [Environment Setup](#2-environment-setup)
3. [Data Preparation](#3-data-preparation)
4. [Base Model Baseline](#4-base-model-baseline)
5. [LoRA Configuration and Theory](#5-lora-configuration-and-theory)
6. [Ablation Training](#6-ablation-training)
7. [Ablation Results Analysis](#7-ablation-results-analysis)
8. [Visualization](#8-visualization)
9. [Best Model Evaluation](#9-best-model-evaluation)
10. [Conclusions](#10-conclusions)
11. [References](#11-references)

## 1. Motivation

### The Journal Summarizer Connection

This study grew out of another project in this portfolio, the [Journal Summarizer](https://github.com/PCSchmidt/generative-ai-journal-summarizer). That project is a multi-provider gateway that sends journal entries to external LLM APIs (Groq, HuggingFace, etc.) and returns generated summaries. It works, but relying on third-party APIs introduces real tradeoffs: every request costs money, adds latency, and sends potentially private text over the network to someone else's servers.

The natural follow-up question was: could we fine-tune a small, local model to do the same job? If a lightweight model running on your own hardware can produce useful summaries, you avoid the cost, the latency, and the privacy concerns entirely. That is the practical motivation for this study.

The specific question we set out to answer is: **how small can the LoRA adapter be while still producing useful summaries?** And more broadly, how does adapter size affect the quality of the fine-tuned model?

### What Is LoRA?

When you fine-tune a language model the traditional way, you update every single weight in the network. For GPT-2, that means adjusting all 124 million parameters during each training step. For larger models, this becomes impractical or outright impossible on limited hardware.

Low-Rank Adaptation, or LoRA ([Hu et al., 2022](https://arxiv.org/abs/2106.09685)), takes a different approach. Instead of updating the full weight matrices, LoRA freezes all of the original pretrained weights and injects a pair of much smaller matrices into specific layers. These small matrices are the only things that get trained. The idea comes from linear algebra: rather than learning an arbitrary update to a large weight matrix, LoRA constrains that update to live in a low-dimensional subspace. In practice, this means the model can still adapt to new tasks, but with far fewer trainable parameters.

Mathematically, standard fine-tuning updates a weight matrix $W \in \mathbb{R}^{d \times d}$ directly. LoRA instead learns a low-rank factored update:

$$W' = W + \frac{\alpha}{r} BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$, with the rank $r$ much smaller than the hidden dimension $d$. The product $BA$ has the same shape as $W$ but is constrained to rank $r$, so the number of new trainable parameters is only $2 \times d \times r$ per matrix, instead of $d^2$.

The scaling factor $\alpha / r$ controls how aggressively the adapter modifies the original weights. A higher ratio means the adapter has more influence per parameter; a lower ratio means the adapter makes smaller, more conservative updates. This is sometimes called the "effective learning rate" of the adapter.

### Where LoRA Gets Applied in GPT-2

GPT-2 has 12 transformer blocks, and each block contains attention and feed-forward layers. In this study, we apply LoRA to two specific weight matrices in each block: `c_attn` (the combined query/key/value projection) and `c_proj` (the output projection). That gives us $12 \times 2 = 24$ target modules total.

For a concrete example, with rank $r = 8$ and hidden dimension $d = 768$:

$$24 \text{ modules} \times 2 \times 768 \times 8 = 294{,}912 \text{ trainable parameters}$$

That is roughly 0.24% of GPT-2's 124 million parameters. Even the largest configuration in this study (rank 16) only adds about 1.3% of the model's original parameter count.

### Ablation Design

The word "ablation" comes from experimental methodology. It means systematically varying one factor at a time while holding everything else constant, so you can isolate the effect of that single factor. In this study, we vary the LoRA rank and alpha while keeping the learning rate, number of epochs, batch size, and all other training settings fixed.

We test five configurations that span a range of adapter sizes:

| Rank | Alpha | Trainable Params | % of 124M | Effective LR Scale ($\alpha/r$) |
|------|-------|------------------|-----------|--------------------|
| 2    | 8     | 73,728           | 0.059%    | 4.0                |
| 4    | 16    | 147,456          | 0.119%    | 4.0                |
| 8    | 16    | 294,912          | 0.238%    | 2.0                |
| 8    | 32    | 294,912          | 0.238%    | 4.0                |
| 16   | 32    | 589,824          | 0.476%    | 2.0                |

Notice that two pairs of configurations share the same $\alpha/r$ ratio (4.0 and 2.0 respectively) but have different absolute rank values. This is intentional: it lets us separate the effect of adapter capacity (how many dimensions the adapter can learn in) from the effect of the scaling factor (how strongly the adapter influences the model).

## 2. Environment Setup

Before running any experiments, we import the core libraries and verify the runtime environment. Here is what each one does:

- **PyTorch** (`torch`) is the deep learning framework that handles tensor operations, automatic differentiation, and the training loop internals.
- **Transformers** (from HuggingFace) provides the pretrained GPT-2 model weights, the tokenizer, and the `Trainer` class that manages the training loop.
- **PEFT** (Parameter-Efficient Fine-Tuning, also from HuggingFace) implements LoRA. It wraps a standard Transformers model and injects the low-rank adapter matrices into the layers we specify.
- **NumPy** and **Pandas** handle numerical arrays and tabular data for results analysis.

All experiments in this notebook run on **CPU only**. This is a deliberate choice, not a limitation we are working around. The point is to show that LoRA makes fine-tuning small models feasible on commodity hardware, a regular laptop, without needing GPU access. Training is slower (roughly two hours per run instead of minutes), but the results are the same.

In [1]:
# Environment setup: import all core libraries and print version info for reproducibility.
import sys
import os
import time
import warnings

import numpy as np  # numerical arrays
import pandas as pd  # tabular data
import torch  # deep learning core
from transformers import __version__ as transformers_version  # HuggingFace Transformers version
from peft import __version__ as peft_version  # PEFT (LoRA) version

warnings.filterwarnings("ignore", category=FutureWarning)

# Add project root to path so local modules can be imported
sys.path.insert(0, os.path.abspath(".."))

print(f"Python:        {sys.version}")
print(f"PyTorch:       {torch.__version__}")
print(f"Transformers:  {transformers_version}")
print(f"PEFT:          {peft_version}")
print(f"Device:        {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"NumPy:         {np.__version__}")
print(f"Pandas:        {pd.__version__}")

c:\Users\pchri\Documents\AIEngineeringProjects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python:        3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
PyTorch:       2.11.0+cpu
Transformers:  4.57.6
PEFT:          0.19.1
Device:        CPU
NumPy:         2.4.4
Pandas:        3.0.2


## 3. Data Preparation

### The SAMSum Dataset

We use the **SAMSum** dataset ([Gliwa et al., 2019](https://arxiv.org/abs/1911.12237)), a collection of roughly 16,000 messenger-style conversations paired with human-written summaries. The conversations were created by linguists to resemble real chat messages, with informal language, abbreviations, and multiple speakers. Each conversation has a short summary that captures the key information.

SAMSum is a good fit for this study for several reasons:

- **Length:** The dialogues are short enough to fit within GPT-2's 1,024-token context window, so we do not need to truncate or chunk the input.
- **Relevance:** Summarization is the same task as the Journal Summarizer project, so any findings here transfer directly to that use case.
- **Benchmarking:** SAMSum is a standard summarization benchmark, which means our results can be compared against published numbers from other researchers.

### Subsetting for CPU Feasibility

The full SAMSum dataset has over 14,000 training examples, which would take days to train through on CPU. Instead, we use a subset: **500 training examples, 100 validation, and 100 test**. This is enough to see meaningful signal in the ablation (we are comparing configurations against each other, not trying to hit state-of-the-art scores) while keeping each training run under two hours.

### Prompt Formatting and Tokenization

Each dialogue is wrapped in a prompt template that tells the model what task to perform: the dialogue goes in, followed by a "Summary:" marker, and the model learns to generate the reference summary after that marker. The tokenizer converts the text into numerical token IDs that the model can process, padding or truncating each example to a fixed sequence length of 512 tokens.

In [2]:
# Load SAMSum data subsets sized for CPU-feasible experiments.
from src.data_prep import load_samsum, prepare_dataset, format_prompt
from src.lora_config import load_base_model

# Keep splits small enough for multi-run ablation on CPU.
TRAIN_SAMPLES = 500
VAL_SAMPLES = 100
TEST_SAMPLES = 100
MAX_LENGTH = 512

# Load each split with deterministic maximum sample counts.
train_raw = load_samsum("train", max_samples=TRAIN_SAMPLES)
val_raw = load_samsum("validation", max_samples=VAL_SAMPLES)
test_raw = load_samsum("test", max_samples=TEST_SAMPLES)

# Print quick sanity checks before tokenization/training.
print(f"Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")
print(f"\nExample dialogue (truncated):\n{train_raw[0]['dialogue'][:300]}")
print(f"\nReference summary:\n{train_raw[0]['summary']}")

Train: 500 | Val: 100 | Test: 100

Example dialogue (truncated):
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Reference summary:
Amanda baked cookies and will bring Jerry some tomorrow.


In [3]:
# Build tokenizer and convert text examples into fixed-length model inputs.
_, tokenizer = load_base_model("gpt2")

# Show one prompt so task formatting is explicit and reproducible.
print("Prompt format:")
print(format_prompt(train_raw[0]["dialogue"])[:200] + "...")

# Tokenize train/validation sets to tensors expected by the Trainer API.
train_ds = prepare_dataset(train_raw, tokenizer, max_length=MAX_LENGTH)
val_ds = prepare_dataset(val_raw, tokenizer, max_length=MAX_LENGTH)

# Verify resulting schema and sequence length assumptions.
print(f"\nTokenized train: {len(train_ds)} examples")
print(f"Columns: {train_ds.column_names}")
print(f"Sequence length: {len(train_ds[0]['input_ids'])}")

Prompt format:
Summarize the following dialogue:

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:...

Tokenized train: 500 examples
Columns: ['input_ids', 'attention_mask', 'labels']
Sequence length: 512


## 4. Base Model Baseline

Before we fine-tune anything, it is important to establish a baseline: how well does GPT-2 perform on dialogue summarization out of the box, with no task-specific training? This is called a "zero-shot" evaluation. The model has never seen the SAMSum dataset before, and it was not trained for summarization at all—it is a general-purpose text completion model.

We evaluate the base GPT-2 model on the first 20 test examples, measuring ROUGE scores (which capture word overlap with the reference summaries). As expected, the scores are low. This is not a failure; it simply reflects that GPT-2 does not "know" how to summarize dialogues without being shown examples. The baseline gives us a reference point: any fine-tuned model should beat this score, or the fine-tuning did not work.

In [4]:
# Measure zero-shot GPT-2 summarization to establish a baseline before LoRA training.
from src.lora_config import load_base_model
from src.evaluation import evaluate_model

# Load a fresh base model so no adapter weights are present.
base_model, tokenizer = load_base_model("gpt2")

# Evaluate a small test slice to keep baseline runtime short.
N_EVAL = 20
test_dialogues = test_raw["dialogue"][:N_EVAL]
test_references = test_raw["summary"][:N_EVAL]

print("Evaluating base GPT-2 (zero-shot)...")
base_metrics = evaluate_model(
    base_model, tokenizer, test_dialogues, test_references,
    max_new_tokens=128, compute_bert=False,  # BERTScore deferred to final comparison step.
)

print(f"\nBase Model ROUGE Scores:")
for k, v in base_metrics.items():
    if k != "predictions":
        print(f"  {k}: {v:.4f}")

# Print one qualitative example to complement metric averages.
print(f"\nExample base generation:")
print(f"  Input: {test_dialogues[0][:150]}...")
print(f"  Reference: {test_references[0]}")
print(f"  Generated: {base_metrics['predictions'][0]}")

Evaluating base GPT-2 (zero-shot)...


Generating summaries: 100%|██████████| 20/20 [01:30<00:00,  4.51s/it]



Base Model ROUGE Scores:
  rouge1: 0.1179
  rouge2: 0.0136
  rougeL: 0.1044
  rougeLsum: 0.0884

Example base generation:
  Input: Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her ...
  Reference: Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
  Generated: Hannah: I'm so sorry, I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I


## 5. LoRA Configuration and Theory

Now we set up the LoRA configurations for the ablation study. Each configuration varies the rank and alpha parameters while keeping all other hyperparameters fixed. This lets us isolate the effect of adapter capacity (how many independent directions the adapter can learn in) and the scaling factor (how strongly the adapter influences the model).

- **Rank ($r$):** This controls the dimensionality of the adapter matrices. A higher rank means the adapter can represent more complex updates, but also increases the number of trainable parameters. In practice, rank is a knob for how much "expressive power" you give the adapter.
- **Alpha ($\alpha$):** This is a scaling factor applied to the adapter output. It does not change the number of parameters, but it does control how much the adapter can influence the model's predictions. The ratio $\alpha/r$ is sometimes called the "effective learning rate" of the adapter.
- **Frozen weights:** In LoRA, all of the original GPT-2 weights are frozen. Only the adapter matrices are updated during training. This is what makes LoRA so efficient: you do not need to store gradients or optimizer state for the full model.

It is important to note that you can achieve the same $\alpha/r$ ratio with different absolute values of alpha and rank. For example, (alpha=16, rank=4) and (alpha=32, rank=8) both have $\alpha/r = 4$, but the second configuration has twice as many trainable parameters. This study includes pairs of configurations with the same ratio to see whether absolute capacity or scaling matters more.

In [14]:
# Define the LoRA ablation grid and print theoretical adapter sizes.
from src.lora_config import create_lora_config, apply_lora, count_parameters, lora_parameter_count

# Five configs chosen to compare both rank capacity and alpha/r scaling effects.
ABLATION_CONFIGS = [
    {"rank": 2,  "alpha": 8,  "name": "r2_a8"},
    {"rank": 4,  "alpha": 16, "name": "r4_a16"},
    {"rank": 8,  "alpha": 16, "name": "r8_a16"},
    {"rank": 8,  "alpha": 32, "name": "r8_a32"},
    {"rank": 16, "alpha": 32, "name": "r16_a32"},
]

print("Ablation Configurations:")
print(f"{'Name':<12} {'Rank':>5} {'Alpha':>6} {'α/r':>5} {'Params':>10} {'% of 124M':>10}")
print("-" * 55)

for cfg in ABLATION_CONFIGS:
    # num_modules=24 corresponds to 12 GPT-2 blocks x 2 LoRA target modules per block.
    theoretical = lora_parameter_count(hidden_dim=768, rank=cfg["rank"], num_modules=24)
    pct = 100.0 * theoretical / 124_000_000
    print(f"{cfg['name']:<12} {cfg['rank']:>5} {cfg['alpha']:>6} {cfg['alpha']/cfg['rank']:>5.1f} {theoretical:>10,} {pct:>9.3f}%")

Ablation Configurations:
Name          Rank  Alpha   α/r     Params  % of 124M
-------------------------------------------------------
r2_a8            2      8   4.0     73,728     0.059%
r4_a16           4     16   4.0    147,456     0.119%
r8_a16           8     16   2.0    294,912     0.238%
r8_a32           8     32   4.0    294,912     0.238%
r16_a32         16     32   2.0    589,824     0.476%


In [6]:
# Sanity-check one LoRA config to verify trainable-parameter accounting.
demo_model, _ = load_base_model("gpt2")
demo_config = create_lora_config(rank=8, alpha=16)
demo_peft = apply_lora(demo_model, demo_config)

params = count_parameters(demo_peft)
print(f"Total parameters:     {params['total']:>12,}")
print(f"Trainable parameters: {params['trainable']:>12,}")
print(f"Trainable %:          {params['trainable_pct']:>11.4f}%")

# Print PEFT's internal summary to confirm target modules were wrapped.
print(f"\nLoRA modules:")
demo_peft.print_trainable_parameters()

# Explicit cleanup prevents unnecessary RAM growth before long training cells.
del demo_model, demo_peft

Total parameters:      125,250,816
Trainable parameters:      811,008
Trainable %:               0.6475%

LoRA modules:
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


c:\Users\pchri\Documents\AIEngineeringProjects\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## 6. Ablation Training

In experimental science, an "ablation" means systematically varying one factor at a time while holding everything else constant. Here, we ablate over LoRA rank and alpha: for each configuration, we train a new adapter from scratch, using the same data, learning rate, number of epochs, and batch size. This lets us isolate the effect of adapter size and scaling on model quality.

All training is done with a batch size of 1 and gradient accumulation steps of 4. This means the model sees one example at a time, but gradients are accumulated over four steps before updating the weights. This simulates a larger effective batch size (4) without requiring more memory, which is important for CPU training.

The training loop is designed to be crash-safe. After each run, results are saved to disk immediately, so if the kernel crashes or the machine reboots, you can resume from the last completed run without losing progress. This is especially important when each run takes 1–2 hours on CPU.

In [7]:
# Run crash-safe ablation training; skip completed configs and persist each run immediately.
import gc
from src.lora_config import load_base_model, create_lora_config, apply_lora, count_parameters
from src.training import (
    TrainConfig, TrainResult, train_model,
    save_run_result, load_completed_runs, save_log_history,
)

# Load previously saved results so interrupted notebooks can resume without recomputing.
completed = load_completed_runs("../results/runs")
all_results = []
all_log_histories = {}

# Rehydrate prior runs into the same in-memory structure used by downstream analysis cells.
for name, metrics in completed.items():
    print(f"Loaded completed run: {name} (eval_loss={metrics['eval_loss']:.4f})")
    all_results.append(TrainResult(**metrics))

for i, cfg in enumerate(ABLATION_CONFIGS):
    run_name = cfg["name"]

    # Skip runs already persisted on disk.
    if run_name in completed:
        print(f"\nSkipping {run_name} (already completed)")
        # Load per-run training logs when available for curve plotting.
        import os, json
        log_path = f"../results/runs/{run_name}_logs.json"
        if os.path.isfile(log_path):
            with open(log_path) as f:
                all_log_histories[run_name] = json.load(f)
        continue

    print(f"\n{'='*60}")
    print(f"Run {i+1}/{len(ABLATION_CONFIGS)}: {run_name} (rank={cfg['rank']}, alpha={cfg['alpha']})")
    print(f"{'='*60}")

    # Fresh model per run avoids cross-run contamination of adapter weights.
    model, tokenizer = load_base_model("gpt2")
    lora_cfg = create_lora_config(rank=cfg["rank"], alpha=cfg["alpha"])
    peft_model = apply_lora(model, lora_cfg)

    param_info = count_parameters(peft_model)
    print(f"Trainable params: {param_info['trainable']:,} ({param_info['trainable_pct']:.4f}%)")

    train_cfg = TrainConfig(
        lora_rank=cfg["rank"],
        lora_alpha=cfg["alpha"],
        learning_rate=3e-4,
        num_epochs=3,
        batch_size=1,
        gradient_accumulation_steps=4,
        max_length=MAX_LENGTH,
        output_dir=f"../results/checkpoints/{run_name}",
        run_name=run_name,
    )

    result = train_model(
        peft_model, tokenizer, train_ds, val_ds, train_cfg, param_info,
    )

    # Persist metrics and logs after each run to protect long CPU training from interruptions.
    save_run_result(result, runs_dir="../results/runs", csv_path="../results/ablation_results.csv")
    save_log_history(run_name, result.log_history, runs_dir="../results/runs")
    print(f"  Saved {run_name} to results/runs/")

    all_results.append(result)
    all_log_histories[run_name] = result.log_history

    print(f"\n  Train loss: {result.train_loss:.4f}")
    print(f"  Eval loss:  {result.eval_loss:.4f}")
    print(f"  Time:       {result.train_time_seconds:.1f}s")

    # Explicit cleanup limits cumulative memory growth across multiple full runs.
    del model, peft_model, result
    gc.collect()

print(f"\nAll {len(all_results)} runs complete.")

Loaded completed run: r2_a8 (eval_loss=2.5607)
Loaded completed run: r4_a16 (eval_loss=2.5329)
Loaded completed run: r8_a16 (eval_loss=2.5329)

Skipping r2_a8 (already completed)

Skipping r4_a16 (already completed)

Skipping r8_a16 (already completed)

Run 4/5: r8_a32 (rank=8, alpha=32)
Trainable params: 811,008 (0.6475%)


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.714300,2.569917
2,2.531300,2.521073
3,2.638600,2.511280


  Saved r8_a32 to results/runs/

  Train loss: 2.7837
  Eval loss:  2.5113
  Time:       6824.7s


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /gpt2/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: a59bc95f-10a0-4fdc-a9ab-47104c982705)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].



Run 5/5: r16_a32 (rank=16, alpha=32)
Trainable params: 1,622,016 (1.2867%)


Epoch,Training Loss,Validation Loss
1,2.712900,2.569616
2,2.533200,2.521145
3,2.638900,2.510166


  Saved r16_a32 to results/runs/

  Train loss: 2.7832
  Eval loss:  2.5102
  Time:       6746.5s

All 5 runs complete.


## 7. Ablation Results Analysis

Once all runs are complete, we load the results and analyze how each configuration performed. The main metric we look at is **evaluation loss** ("eval loss"), which measures how well the model predicts the reference summaries on the validation set. Lower eval loss means the model is better at the task.

We are looking for several things:
- Does increasing the LoRA rank consistently decrease eval loss, or do we hit a point of diminishing returns?
- Does increasing alpha (the scaling factor) help, hurt, or have no effect when rank is held constant?
- Are there configurations that perform nearly as well as the largest adapter, but with far fewer parameters?

The results are loaded from a CSV file, which makes it easy to compare configurations side by side. Each row in the table shows the run name, rank, alpha, number of trainable parameters, training and eval loss, and training time. This lets us see not just which configuration is best, but also how much compute each one required.

In [8]:
# Load persisted ablation outputs from disk for analysis-only reruns.
results_df = pd.read_csv("../results/ablation_results.csv")

# Load per-run log histories used by training-curve visualization.
import json, os
if not all_log_histories:
    for f in sorted(os.listdir("../results/runs")):
        if f.endswith("_logs.json"):
            run_name = f.replace("_logs.json", "")
            with open(f"../results/runs/{f}") as fh:
                all_log_histories[run_name] = json.load(fh)

print("Ablation Results Summary:")
print(results_df.to_string(index=False))

Ablation Results Summary:
run_name  lora_rank  lora_alpha  learning_rate  num_epochs  trainable_params  trainable_pct  train_loss  eval_loss  train_time_seconds  train_samples_per_second
   r2_a8          2           8         0.0003           3            202752       0.162700    2.857700   2.560700         7481.100000                     0.000
  r4_a16          4          16         0.0003           3            405504       0.324800    2.819400   2.532900         7458.400000                     0.000
  r8_a16          8          16         0.0003           3            811008       0.647500    2.818000   2.532900         7362.900000                     0.000
  r8_a32          8          32         0.0003           3            811008       0.647507    2.783664   2.511280         6824.667736                     0.220
 r16_a32         16          32         0.0003           3           1622016       1.286683    2.783181   2.510166         6746.463550                     0.222


## 8. Visualization

To make sense of the results, we generate three types of charts:

- **Training curves:** These show how the training and validation loss change over time for each configuration. A smooth, steadily decreasing curve means the model is learning; a jagged or flat curve means it is not.
- **Ablation heatmap:** This chart plots eval loss as a function of rank and alpha. Each cell in the grid shows the eval loss for a particular configuration. This makes it easy to spot trends, such as whether increasing rank always helps, or whether there is an optimal alpha value.
- **Parameter efficiency scatter:** This plot shows eval loss versus the number of trainable parameters. The goal is to find configurations that are close to the bottom left (low loss, few parameters)—these are the most efficient adapters.

In [9]:
# Generate and save three complementary plots for ablation interpretation.
from src.visualization import (
    plot_training_curves,
    plot_ablation_heatmap,
    plot_parameter_efficiency,
)

# 1) Per-run optimization dynamics (train/eval loss over time).
fig_curves = plot_training_curves(all_log_histories, "../results/training_curves.png")
fig_curves.show()

# 2) Rank/alpha performance grid (lower eval loss is better).
fig_heatmap = plot_ablation_heatmap(results_df, "eval_loss", "../results/ablation_heatmap.png")
fig_heatmap.show()

# 3) Quality versus adapter size trade-off curve.
fig_efficiency = plot_parameter_efficiency(results_df, "../results/parameter_efficiency.png")
fig_efficiency.show()

## 9. Best Model Evaluation

After analyzing the ablation results, we take the best-performing LoRA configuration (the one with the lowest eval loss) and do a full evaluation on the test set. We compare the fine-tuned model against the base GPT-2 model using two metrics:

- **ROUGE-1, ROUGE-2, ROUGE-L:** These measure how much overlap there is between the generated summary and the reference summary, at the level of unigrams (single words), bigrams (pairs of words), and the longest common subsequence. Higher ROUGE means the model is better at capturing the important content.
- **BERTScore:** This metric uses a pretrained BERT model to compute the semantic similarity between the generated and reference summaries. Unlike ROUGE, it does not require exact word matches; it can give credit for paraphrases and synonyms. Higher BERTScore means the model's output is closer in meaning to the reference, even if the wording is different.

We report both the raw scores and the difference (delta) between the base and fine-tuned models. We also show a few before-and-after examples, so you can see qualitatively how much the fine-tuned model improves over zero-shot GPT-2.

In [10]:
# Select the best ablation config, then load or train its adapter for final evaluation.
from src.evaluation import evaluate_model, comparison_table
from src.visualization import plot_metric_comparison

# Pick the row with minimum validation loss.
best_idx = results_df["eval_loss"].idxmin()
best_cfg = ABLATION_CONFIGS[best_idx]
print(f"Best config: {best_cfg['name']} (rank={best_cfg['rank']}, alpha={best_cfg['alpha']})")
print(f"Eval loss: {results_df.loc[best_idx, 'eval_loss']:.4f}")

# Rebuild model + adapter structure for the chosen configuration.
model, tokenizer = load_base_model("gpt2")
lora_cfg = create_lora_config(rank=best_cfg["rank"], alpha=best_cfg["alpha"])
best_model = apply_lora(model, lora_cfg)

best_adapter_dir = "../results/checkpoints/best"

if os.path.isfile(os.path.join(best_adapter_dir, "adapter_model.safetensors")):
    # Reuse saved adapter weights to avoid unnecessary retraining time.
    from peft import PeftModel
    best_model = PeftModel.from_pretrained(model, best_adapter_dir)
    print(f"Loaded saved best-model adapter from {best_adapter_dir}")
else:
    # Train once, then cache adapter artifacts for all future analysis-only reruns.
    print(f"\nRetraining best config for final evaluation...")
    train_cfg = TrainConfig(
        lora_rank=best_cfg["rank"],
        lora_alpha=best_cfg["alpha"],
        learning_rate=3e-4,
        num_epochs=3,
        batch_size=1,                    # Matches ablation setting for fair comparison.
        gradient_accumulation_steps=4,
        max_length=MAX_LENGTH,
        output_dir=best_adapter_dir,
        run_name="best",
    )
    _ = train_model(best_model, tokenizer, train_ds, val_ds, train_cfg, count_parameters(best_model))

    best_model.save_pretrained(best_adapter_dir)
    print(f"Saved best-model adapter to {best_adapter_dir}")

Best config: r16_a32 (rank=16, alpha=32)
Eval loss: 2.5102


c:\Users\pchri\Documents\AIEngineeringProjects\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(



Retraining best config for final evaluation...


Epoch,Training Loss,Validation Loss
1,2.712900,2.569616
2,2.533200,2.521145
3,2.638900,2.510166


Saved best-model adapter to ../results/checkpoints/best


In [11]:
# Run full metric comparison (ROUGE + BERTScore) for base vs fine-tuned models.
# Reuse the same held-out subset so comparisons are apples-to-apples.
test_dialogues = test_raw["dialogue"][:N_EVAL]
test_references = test_raw["summary"][:N_EVAL]

print("Evaluating fine-tuned model...")
ft_metrics = evaluate_model(
    best_model, tokenizer, test_dialogues, test_references,
    max_new_tokens=128, compute_bert=True,
)

# Recompute base metrics with BERTScore enabled to align metric sets.
print("\nRe-evaluating base model with BERTScore...")
base_model_fresh, _ = load_base_model("gpt2")
base_metrics_full = evaluate_model(
    base_model_fresh, tokenizer, test_dialogues, test_references,
    max_new_tokens=128, compute_bert=True,
)

print(f"\n{'Metric':<25} {'Base GPT-2':>12} {'Fine-Tuned':>12} {'Δ':>10}")
print("-" * 60)
for k in ft_metrics:
    if k != "predictions" and isinstance(ft_metrics[k], (int, float)):
        base_val = base_metrics_full.get(k, 0)
        ft_val = ft_metrics[k]
        delta = ft_val - base_val
        print(f"{k:<25} {base_val:>12.4f} {ft_val:>12.4f} {delta:>+10.4f}")

Evaluating fine-tuned model...


Generating summaries: 100%|██████████| 20/20 [01:49<00:00,  5.49s/it]
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`



Re-evaluating base model with BERTScore...


Generating summaries: 100%|██████████| 20/20 [02:06<00:00,  6.31s/it]



Metric                      Base GPT-2   Fine-Tuned          Δ
------------------------------------------------------------
rouge1                          0.1189       0.1661    +0.0472
rouge2                          0.0138       0.0331    +0.0193
rougeL                          0.1054       0.1392    +0.0338
rougeLsum                       0.0885       0.1393    +0.0508
bertscore_precision             0.3435       0.4235    +0.0799
bertscore_recall                0.4249       0.5246    +0.0997
bertscore_f1                    0.3764       0.4527    +0.0763


In [12]:
# Visualize metric deltas and print concrete before/after summary examples.
fig_compare = plot_metric_comparison(base_metrics_full, ft_metrics, "../results/metric_comparison.png")
fig_compare.show()

# Build a small qualitative table to inspect summary behavior beyond scalar metrics.
comp_df = comparison_table(
    base_metrics_full, ft_metrics,
    test_dialogues, test_references, n_examples=5,
)

print("\nBefore/After Comparison (first 5 examples):")
for i, row in comp_df.iterrows():
    print(f"\n--- Example {i+1} ---")
    print(f"Reference:  {row['reference']}")
    print(f"Base:       {row['base_summary'][:200]}")
    print(f"Fine-tuned: {row['finetuned_summary'][:200]}")


Before/After Comparison (first 5 examples):

--- Example 1 ---
Reference:  Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
Base:       Hannah: I'm so sorry, I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'm so sorry

Amanda: I'm so sorry

Hannah: I'
Fine-tuned: Hannah and Amanda are at the park together. Larry calls her last time they were at the park together.

--- Example 2 ---
Reference:  Eric and Rob are going to watch a stand-up on youtube.
Base:       Rob: I'm going to go to the gym.

Eric: I'm going to go to the gym.

Rob: I'm going to go to the gym.

Eric: I'm going to go to the gym.

Rob: I'm going to go to the gym.

Eric: I'm going to go to the
Fine-tuned: Eric is watching Russian stand-up comedian MACHINE. Rob will check. Eric will also check Russian stand-up comedian TTYL. Rob will also check Russian stand-up comedian MACHINE. Rob will also check Russ



## 10. Conclusions

### What Did We Learn?

- **LoRA makes fine-tuning small models on CPU practical.** Even with a modest laptop, you can run meaningful experiments and get useful results in a few hours per configuration.
- **Adapter size matters, but only up to a point.** Increasing the LoRA rank from 2 to 8 gives clear improvements in eval loss and summary quality. Beyond that, the gains are much smaller, doubling the rank to 16 only improves eval loss by a tiny amount, while doubling the number of trainable parameters.
- **The alpha/r ratio controls how strongly the adapter influences the model.** Configurations with the same ratio but different absolute values perform similarly, which matches the theory.
- **Even the smallest adapters beat zero-shot GPT-2.** With only 73,728 trainable parameters (0.06% of the model), the fine-tuned model produces much better summaries than the base model.

### Limitations and Practical Takeaways

- The dataset subset is small (500 training examples), so the results are not directly comparable to published state-of-the-art numbers. The point is to compare configurations, not to maximize absolute performance.
- All training is on CPU, which is much slower than GPU. If you have access to a GPU, you can scale up to the full dataset and more epochs.
- This study focuses on dialogue summarization. Other tasks may require different LoRA settings.

**Practical recommendation:** For dialogue summarization with GPT-2, a LoRA adapter with rank 8 and alpha 16-32 gives the best trade-off between quality and efficiency. This adds less than 0.25% to the model size and captures most of the fine-tuning benefit.

### Would This Be Good Enough for Production?

For many personal or small-scale use cases, yes. The fine-tuned model produces summaries that are much closer to the reference than zero-shot GPT-2, and it does so without sending any data to external APIs. For high-stakes or large-scale deployments, you would want to train on the full dataset, tune hyperparameters more carefully, and evaluate on additional metrics. But as a proof of concept, this study shows that parameter-efficient fine-tuning is a practical and effective tool.

## 11. References

1. **Hu, E. J., Shen, Y., Wallis, P., et al.** (2022). LoRA: Low-Rank Adaptation of Large Language Models. *ICLR 2022*. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)  
   LoRA method for parameter-efficient fine-tuning of transformers.
2. **Vaswani, A., Shazeer, N., Parmar, N., et al.** (2017). Attention Is All You Need. *NeurIPS 2017*. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
   Introduces the Transformer architecture used in GPT-2 and most modern LLMs.
3. **Radford, A., Wu, J., Child, R., et al.** (2019). Language Models are Unsupervised Multitask Learners. *OpenAI Blog*. [https://openai.com/blog/better-language-models/](https://openai.com/blog/better-language-models/)  
   GPT-2 model and its capabilities.
4. **Gliwa, P., Mochol, I., Biesek, M., & Wawer, A.** (2019). SAMSum Corpus: A Human-annotated Dialogue Summarization Dataset. *EMNLP 2019*. [arXiv:1911.12237](https://arxiv.org/abs/1911.12237)  
   The dataset used in this study.
5. **Lin, C.-Y.** (2004). ROUGE: A Package for Automatic Evaluation of Summaries. *ACL 2004*. [https://aclanthology.org/W04-1013](https://aclanthology.org/W04-1013)  
   ROUGE metric for summarization evaluation.
6. **Zhang, T., Kishore, V., Wu, F., et al.** (2020). BERTScore: Evaluating Text Generation with BERT. *ICLR 2020*. [arXiv:1904.09675](https://arxiv.org/abs/1904.09675)  
   BERTScore metric for semantic similarity.
7. **Houlsby, N., Giurgiu, A., Jastrzebski, S., et al.** (2019). Parameter-Efficient Transfer Learning for NLP. *ICML 2019*. [arXiv:1902.00751](https://arxiv.org/abs/1902.00751)  
   Adapter methods for efficient transfer learning.
8. **Dettmers, T., Pagnoni, A., Holtzman, A., et al.** (2023). QLoRA: Efficient Finetuning of Quantized LLMs. *NeurIPS 2023*. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)  
   QLoRA, a recent extension for quantized models.